In [ ]:
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score


# Get the data
d = load_wine()
X, y = d.data, d.target


# 1. Stratified train-test split
def stratified_split(X, y, test_size=0.2, seed=0):
    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=test_size,
        random_state=seed,
        stratify=y
    )

    return X_train, X_test, y_train, y_test


# 2. Build pipeline
def build_pipeline():
    pipe = Pipeline([
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(max_iter=5000))
    ])

    return pipe


# 3. Stratified K-Fold cross-validation
def kfold_score(pipe, X, y, k=5):
    cv = StratifiedKFold(
        n_splits=k,
        shuffle=True,
        random_state=0
    )

    scores = cross_val_score(
        pipe,
        X,
        y,
        cv=cv,
        scoring="accuracy"
    )

    return round(float(scores.mean()), 3)


# 4. Honest held-out test score
def honest_test_score(pipe, X, y):
    X_train, X_test, y_train, y_test = stratified_split(
        X,
        y,
        test_size=0.2,
        seed=0
    )

    pipe.fit(X_train, y_train)

    y_pred = pipe.predict(X_test)

    score = accuracy_score(y_test, y_pred)

    return round(float(score), 3)


# Verify
pipe = build_pipeline()

print(kfold_score(pipe, X, y, 5))
print(honest_test_score(build_pipeline(), X, y))